In [7]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

# V3 FEATURE ENGINEERING

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

FINAL_DIR = ROOT / "data" / "final"
RESULTS_DIR = ROOT / "outputs" / "results"
FINAL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

input_path = FINAL_DIR / "ml_ready_player_games.parquet"
output_path = FINAL_DIR / "ml_ready_player_games_v3.parquet"

df = pd.read_parquet(input_path)
df["game_date"] = pd.to_datetime(df["game_date"])
df = df.sort_values(["player_id", "game_date"]).copy()

# Rest / fatigue
df["days_since_last_game"] = df.groupby("player_id")["game_date"].diff().dt.days.fillna(3)
df["is_back_to_back"] = (df["days_since_last_game"] == 1).astype(int)
df["is_short_rest"] = (df["days_since_last_game"] <= 2).astype(int)

# Trend features
df["pts_trend_l5_l10"] = df["l5_avg_pts"] - df["l10_avg_pts"]
df["reb_trend_l5_l10"] = df["l5_avg_reb"] - df["l10_avg_reb"]
df["ast_trend_l5_l10"] = df["l5_avg_ast"] - df["l10_avg_ast"]

# Opponent relative context
league_avg_pace = df["opponent_pace"].mean()
league_avg_def = df["opponent_def_rating"].mean()
df["opp_pace_rel"] = df["opponent_pace"] - league_avg_pace
df["opp_def_rating_rel"] = df["opponent_def_rating"] - league_avg_def

# Pace-adjusted production
df["pace_adj_pts"] = df["season_avg_pts"] * (df["opponent_pace"] / league_avg_pace)
df["pace_adj_reb"] = df["season_avg_reb"] * (df["opponent_pace"] / league_avg_pace)
df["pace_adj_ast"] = df["season_avg_ast"] * (df["opponent_pace"] / league_avg_pace)

# Usage / opportunity interactions
df["minutes_x_usage"] = df["season_avg_min"] * df["season_avg_usg"]
df["usage_x_opp_pace"] = df["season_avg_usg"] * df["opponent_pace"]
df["usage_x_opp_def"] = df["season_avg_usg"] * df["opponent_def_rating"]

if "home_flag" in df.columns:
    df["home_x_usage"] = df["home_flag"] * df["season_avg_usg"]

# Availability intensity
df["missing_pts_share"] = df["missing_pts_l5"] / (df["season_avg_pts"] + 1)
df["missing_reb_share"] = df["missing_reb_l5"] / (df["season_avg_reb"] + 1)
df["missing_ast_share"] = df["missing_ast_l5"] / (df["season_avg_ast"] + 1)

if "teammates_out_count" in df.columns:
    df["teammates_out_x_usage"] = df["teammates_out_count"] * df["season_avg_usg"]
    df["teammates_out_x_min"] = df["teammates_out_count"] * df["season_avg_min"]

# Target-specific matchup interactions
df["pts_l5_x_opp_def"] = df["l5_avg_pts"] * df["opponent_def_rating"]
df["reb_l5_x_opp_def"] = df["l5_avg_reb"] * df["opponent_def_rating"]
df["ast_l5_x_opp_def"] = df["l5_avg_ast"] * df["opponent_def_rating"]

# Cleanup
df = df.replace([np.inf, -np.inf], np.nan).fillna(0)

# Save
df.to_parquet(output_path, index=False)

v3_features = [
    "days_since_last_game", "is_back_to_back", "is_short_rest",
    "pts_trend_l5_l10", "reb_trend_l5_l10", "ast_trend_l5_l10",
    "opp_pace_rel", "opp_def_rating_rel",
    "pace_adj_pts", "pace_adj_reb", "pace_adj_ast",
    "minutes_x_usage", "usage_x_opp_pace", "usage_x_opp_def",
    "missing_pts_share", "missing_reb_share", "missing_ast_share",
    "teammates_out_x_usage", "teammates_out_x_min",
    "pts_l5_x_opp_def", "reb_l5_x_opp_def", "ast_l5_x_opp_def"
]
v3_features = [c for c in v3_features if c in df.columns]

summary = {
    "rows": int(df.shape[0]),
    "columns": int(df.shape[1]),
    "v3_feature_count": len(v3_features),
    "v3_features": v3_features,
}
(RESULTS_DIR / "v3_feature_summary.json").write_text(json.dumps(summary, indent=2))

print("V3 feature engineering complete.")
print("Saved:", output_path)
print("Shape:", df.shape)
print("New V3 features:", len(v3_features))

V3 feature engineering complete.
Saved: c:\Users\aruls\anaconda_projects\analytics\NBACapstone-main\data\final\ml_ready_player_games_v3.parquet
Shape: (52707, 92)
New V3 features: 22
